# BQNT patch: implied volatility and premium in percent

**Apply this to your existing selected-contract table.** It preserves the chosen option IDs, sides, strikes and expiries; it does not implement a new option-selection rule or historical-chain backfill.

Copy the implementation cell and launch cell into the BQNT notebook containing your current output DataFrame. The launch example calls that DataFrame `result`; change that single variable name to your actual variable.

The main view shows **Bloomberg mid implied volatility (%)**, **mid premium / spot (%)**, **mid premium / strike (%)**, **strike / spot (%)**, and **bid–ask spread / mid (%)**. Dollar premium remains available only in audit columns. All new market inputs are requested through `bql.Service()`.

**Status:** percentage arithmetic, unit conversion, missing-data handling, quote/currency/date checks, response parsing and HTML formatting were tested offline with mock responses. Bloomberg requests and BQNT widgets were not tested in a Bloomberg session.

## Unit and price conventions

`iv_units="percent"` explicitly assumes a raw IV value of 35 means 35%. Verify this against Bloomberg's field definition. Use `"decimal"` only when a raw value of 0.35 means 35%. No magnitude-based unit guessing is used.

Premium is `(bid + ask) / 2`; percentage premiums are **not annualized**. Bloomberg `ivol(mid)` is returned as a separate analytic; it is not the average of bid IV and ask IV, and is not independently calibrated here to the arithmetic midpoint displayed in this table.

This implementation is for standard USD equity-option premium/strike units. Confirm standard deliverables in the original chain selection. Missing/mismatched USD currency metadata suppresses premium percentages; crossed/missing quotes are not filled. Bloomberg observation dates are retained separately. Fetch timestamps are not quote timestamps, and matching dates do not establish intraday freshness.

A one-year Z-score remains a separate, dimensionless statistic. This patch does not carry over or rename the previous dollar-premium Z-scores, and does not create missing historical observations.

In [ ]:
"""BQNT pricing/ranking patch: Bloomberg mid IV (%) and premium percentages.

Input: the existing selected-contract DataFrame, with TICKER, SIDE, EXPIRY,
STRIKE, ID. Preserves selections; does not rediscover options or build history.
All fresh market inputs come from bql.Service(); no demo/CSV pricing path.

IV expression: ivol(mid), illustrated by the original OMON example at
https://blog.iqmo.com/blog/bqnt/bquant_e_7_omonapp/
Live Bloomberg execution has not been tested by the author in this environment.
"""
from __future__ import annotations

from html import escape
from typing import Callable
import numpy as np
import pandas as pd

RANK_METRICS = ("IMPLIED VOL PCT", "PREMIUM PCT SPOT", "PREMIUM PCT STRIKE")
PERCENT_COLUMNS = (*RANK_METRICS, "STRIKE PCT SPOT", "SPREAD PCT", "RANK VALUE")
MAIN_COLUMNS = ["TICKER", "SIDE", "EXPIRY", "DTE", "STRIKE PCT SPOT",
                "IMPLIED VOL PCT", "PREMIUM PCT SPOT", "PREMIUM PCT STRIKE",
                "SPREAD PCT", "RANK METRIC", "RANK VALUE", "QUOTE STATUS", "IV STATUS"]


def _number(s: pd.Series) -> pd.Series:
    if not pd.api.types.is_numeric_dtype(s):
        s = s.astype("string").str.replace(",", "", regex=False).replace(
            {"—": pd.NA, "": pd.NA, "N/A": pd.NA})
    return pd.to_numeric(s, errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)


def _selected(rows: pd.DataFrame) -> pd.DataFrame:
    if not isinstance(rows, pd.DataFrame) or rows.empty:
        raise ValueError("Pass your existing nonempty selected-contract DataFrame.")
    d = rows.copy()
    if any(str(n).upper() in {"TICKER", "ID"} for n in d.index.names):
        d = d.reset_index()
    d.columns = [str(c).strip().upper().replace("_", " ") for c in d.columns]
    if d.columns.duplicated().any():
        raise ValueError("Column names become duplicates after normalization.")
    required = ["TICKER", "SIDE", "EXPIRY", "STRIKE", "ID"]
    missing = set(required) - set(d.columns)
    if missing:
        raise ValueError(f"Required columns missing: {sorted(missing)}")
    # Rebuild the quote table rather than carry stale metrics/old z-scores into it.
    d = d[required].copy().reset_index(drop=True)
    for col in ["TICKER", "ID"]:
        if d[col].isna().any():
            raise ValueError(f"Missing {col}.")
        d[col] = d[col].astype(str).str.strip()
        if d[col].eq("").any():
            raise ValueError(f"Blank {col}.")
    d["SIDE"] = d["SIDE"].astype(str).str.upper().str.strip().map(
        {"C": "CALL", "CALL": "CALL", "P": "PUT", "PUT": "PUT"})
    d["STRIKE"] = _number(d["STRIKE"])
    d["EXPIRY"] = pd.to_datetime(d["EXPIRY"], errors="raise")
    if d["EXPIRY"].dt.tz is not None:
        raise ValueError("EXPIRY must be a timezone-naive market date.")
    d["EXPIRY"] = d["EXPIRY"].dt.normalize()
    if d["SIDE"].isna().any() or not d["STRIKE"].gt(0).all() or d["EXPIRY"].isna().any():
        raise ValueError("Invalid SIDE, STRIKE, or EXPIRY.")
    if d.groupby("ID")[["TICKER", "SIDE", "EXPIRY", "STRIKE"]].nunique().gt(1).any().any():
        raise ValueError("An option ID has conflicting contract metadata.")
    return d


def _result_frame(result, alias: str) -> pd.DataFrame:
    f = result.df().reset_index()
    if f.empty:
        return pd.DataFrame(columns=["ID", "VALUE", "DATE", "CURRENCY"])
    lookup = {str(c).strip().upper(): c for c in f.columns}
    value_col = next((lookup[k] for k in [alias.upper(), "#" + alias.upper()]
                      if k in lookup), None)
    if "ID" not in lookup or value_col is None:
        raise ValueError(f"{alias}: unexpected result columns {list(f.columns)}")
    out = pd.DataFrame({"ID": f[lookup["ID"]], "VALUE": _number(f[value_col])})
    out["DATE"] = pd.to_datetime(f[lookup["DATE"]], errors="coerce") if "DATE" in lookup else pd.NaT
    out["CURRENCY"] = f[lookup["CURRENCY"]].astype("string").str.upper() if "CURRENCY" in lookup else pd.NA
    if out["ID"].isna().any() or out["ID"].duplicated().any():
        raise ValueError(f"{alias}: expected one snapshot row per ID; inspect BQL dimensions.")
    out["ID"] = out["ID"].astype(str)
    return out


def _fetch(bq, ids, expressions: dict, batch_size: int, progress: Callable) -> tuple:
    ids = sorted(set(ids))
    pieces = {k: [] for k in expressions}
    errors = []
    for offset in range(0, len(ids), batch_size):
        batch = ids[offset:offset + batch_size]
        # IDs come from the existing Bloomberg DataFrame, never generated tickers.
        if any(any(c in sid for c in "'\\\n\r") for sid in batch):
            raise ValueError("Unexpected quote/backslash/newline in a security identifier.")
        get = ", ".join(f"{expr} as #{alias}" for alias, expr in expressions.items())
        query = f"get({get}) for({repr(batch)})"
        progress(f"{', '.join(expressions)}: {min(offset + batch_size, len(ids))}/{len(ids)}")
        try:
            response = bq.execute(query)
            if len(response) != len(expressions):
                raise ValueError("BQL returned an unexpected number of data items.")
            parsed = {k: _result_frame(r, k) for k, r in zip(expressions, response)}
            for k, frame in parsed.items():
                pieces[k].append(frame)
        except Exception as exc:
            errors.append({"FIELDS": ", ".join(expressions), "IDS": ", ".join(batch), "ERROR": str(exc)})
    frames = {}
    for k, parts in pieces.items():
        f = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(
            columns=["ID", "VALUE", "DATE", "CURRENCY"])
        if f["ID"].duplicated().any():
            raise ValueError(f"{k}: duplicate returned IDs across batches.")
        frames[k] = f.set_index("ID")
    return frames, errors


def calculate_percentages(data: pd.DataFrame, iv_units: str = "percent") -> pd.DataFrame:
    """Values stored in percentage points: 7.671 means 7.671%, not 767.1%.

    iv_units is explicit, not inferred from size. 'percent': raw 35 -> 35%.
    'decimal': raw 0.35 -> 35%. Verify the unit against the BQL field definition.
    This patch is restricted to standard USD equity-option premium/strike units.
    Currency checks use the returned price metadata, not a silently assumed USD.
    """
    if iv_units not in {"percent", "decimal"}:
        raise ValueError("iv_units must be 'percent' or 'decimal'.")
    d = data.copy()
    for c in ["BID", "ASK", "STOCK LAST", "STRIKE", "IV RAW"]:
        d[c] = _number(d[c])
    for c in ["BID CURRENCY", "ASK CURRENCY", "STOCK CURRENCY"]:
        d[c] = d[c].astype("string").str.strip().str.upper()
    currency_ok = d[["BID CURRENCY", "ASK CURRENCY", "STOCK CURRENCY"]].eq("USD").fillna(False).all(axis=1)
    valid_quote = d["BID"].ge(0) & d["ASK"].gt(0) & d["ASK"].ge(d["BID"])
    quote_date_mismatch = (d["BID DATE"].notna() & d["ASK DATE"].notna()
                           & d["BID DATE"].ne(d["ASK DATE"]))
    spot_date_mismatch = (d["BID DATE"].notna() & d["STOCK DATE"].notna()
                          & d["BID DATE"].ne(d["STOCK DATE"]))
    valid_quote &= ~quote_date_mismatch
    premium = ((d["BID"] + d["ASK"]) / 2).where(valid_quote)
    d["PRICE BASIS"] = "MID"
    d["PREMIUM"] = premium  # Audit column: still a currency price, never labelled %.
    d["PREMIUM PCT SPOT"] = (100 * premium / d["STOCK LAST"].where(d["STOCK LAST"].gt(0))).where(currency_ok & ~spot_date_mismatch)
    d["PREMIUM PCT STRIKE"] = (100 * premium / d["STRIKE"].where(d["STRIKE"].gt(0))).where(currency_ok)
    d["STRIKE PCT SPOT"] = (100 * d["STRIKE"] / d["STOCK LAST"].where(d["STOCK LAST"].gt(0))).where(d["STOCK CURRENCY"].eq("USD").fillna(False))
    d["SPREAD PCT"] = (100 * (d["ASK"] - d["BID"]) / premium.where(premium.gt(0))).where(
        d["BID CURRENCY"].eq(d["ASK CURRENCY"]).fillna(False))
    d["IMPLIED VOL PCT"] = (d["IV RAW"] * (100 if iv_units == "decimal" else 1)).where(d["IV RAW"].ge(0))
    d["IV BASIS"] = "Bloomberg IVOL(MID)"
    d["IV UNITS"] = iv_units
    d["QUOTE STATUS"] = "Indicative; intraday quote age unverified"
    d.loc[d["BID"].eq(0), "QUOTE STATUS"] = "Zero bid; indicative midpoint"
    d.loc[spot_date_mismatch, "QUOTE STATUS"] = "Spot/option dates differ; % spot unavailable"
    d.loc[~currency_ok, "QUOTE STATUS"] = "USD currency check failed/missing; premium % unavailable"
    d.loc[quote_date_mismatch, "QUOTE STATUS"] = "Bid/ask dates differ; midpoint unavailable"
    d.loc[~(d["BID"].ge(0) & d["ASK"].gt(0) & d["ASK"].ge(d["BID"])), "QUOTE STATUS"] = "Missing/invalid/crossed bid-ask"
    d["IV STATUS"] = "Bloomberg mid IV; unit convention configured"
    d.loc[d["IMPLIED VOL PCT"].isna(), "IV STATUS"] = "IV unavailable; see query issues"
    iv_date_mismatch = d["IV DATE"].notna() & d["BID DATE"].notna() & d["IV DATE"].ne(d["BID DATE"])
    d.loc[iv_date_mismatch & d["IMPLIED VOL PCT"].notna(), "IV STATUS"] = "IV/quote dates differ; inspect audit columns"
    return d


def refresh_percentage_table(bq, option_rows: pd.DataFrame, iv_units: str = "percent",
                             batch_size: int = 40, progress: Callable = lambda _: None) -> tuple:
    """Refresh current prices/IV for the SAME selections; return (table, errors).

    Native BQL expression ivol(mid) is requested separately, so IV field failures
    do not suppress otherwise valid premiums. No pricing-model fallback is used.
    Prices and IV are not guaranteed simultaneous or from identical model inputs.
    """
    if batch_size < 1:
        raise ValueError("batch_size must be positive")
    d = _selected(option_rows)
    started = pd.Timestamp.now(tz="UTC")
    stock, e1 = _fetch(bq, d["TICKER"], {"STOCK": "px_last()"}, batch_size, progress)
    quotes, e2 = _fetch(bq, d["ID"], {"BID": "px_bid()", "ASK": "px_ask()"}, batch_size, progress)
    iv, e3 = _fetch(bq, d["ID"], {"IV_RAW": "ivol(mid)"}, batch_size, progress)
    for source, name, key, value_col in [(stock, "STOCK", "TICKER", "STOCK LAST"),
                                       (quotes, "BID", "ID", "BID"),
                                       (quotes, "ASK", "ID", "ASK"),
                                       (iv, "IV_RAW", "ID", "IV RAW")]:
        f = source[name]
        label = "IV" if name == "IV_RAW" else name
        d[value_col] = d[key].map(f["VALUE"])
        d[f"{label} DATE"] = pd.to_datetime(d[key].map(f["DATE"]), errors="coerce")
        d[f"{label} CURRENCY"] = d[key].map(f["CURRENCY"])
    valuation_date = started.tz_convert("America/New_York").tz_localize(None).normalize()
    d["VALUATION DATE"] = valuation_date
    d["DTE"] = (d["EXPIRY"] - valuation_date).dt.days
    d["FETCH STARTED UTC"] = started.isoformat()
    d["FETCH COMPLETED UTC"] = pd.Timestamp.now(tz="UTC").isoformat()
    d = calculate_percentages(d, iv_units)
    expired = d["DTE"].lt(0)
    d.loc[expired, list(RANK_METRICS)] = np.nan
    d.loc[expired, ["QUOTE STATUS", "IV STATUS"]] = "Expired selection; rerun contract selection"
    return rank_table(d), pd.DataFrame(e1 + e2 + e3, columns=["FIELDS", "IDS", "ERROR"])


def rank_table(data: pd.DataFrame, metric: str = "PREMIUM PCT SPOT") -> pd.DataFrame:
    if metric not in RANK_METRICS:
        raise ValueError(f"Choose one of {RANK_METRICS}; dollar premium is not a rank metric.")
    d = data.copy()
    d["RANK METRIC"] = metric
    d["RANK VALUE"] = d[metric]
    return d.sort_values("RANK VALUE", ascending=False, na_position="last", kind="mergesort").reset_index(drop=True)


def display_table(data: pd.DataFrame, audit: bool = False):
    from IPython.display import HTML, display
    d = data.copy() if audit else data.reindex(columns=MAIN_COLUMNS).copy()
    for col in d:
        if col in PERCENT_COLUMNS:
            # Stored values are already in percentage points. Do not use :.3%.
            d[col] = d[col].map(lambda x: "—" if pd.isna(x) else f"{x:,.3f}%")
        elif pd.api.types.is_datetime64_any_dtype(d[col]):
            d[col] = d[col].dt.strftime("%Y-%m-%d").fillna("—")
    display(HTML('<div style="overflow:auto;max-height:620px">' +
                 d.to_html(index=False, escape=True, na_rep="—", float_format=lambda x: f"{x:,.3f}") + '</div>'))


def launch_dashboard(bq, option_rows: pd.DataFrame, iv_units: str = "percent"):
    """Notebook widget; click Refresh Bloomberg after display()."""
    import ipywidgets as w
    from IPython.display import HTML, display
    selected = _selected(option_rows)
    rank = w.Dropdown(options=[("Implied vol (%)", "IMPLIED VOL PCT"),
                              ("Premium (% spot)", "PREMIUM PCT SPOT"),
                              ("Premium (% strike)", "PREMIUM PCT STRIKE")],
                      value="PREMIUM PCT SPOT", description="Rank by:")
    side = w.Dropdown(options=["ALL", "CALL", "PUT"], description="Side:")
    search = w.Text(description="Ticker:", continuous_update=False)
    limit = w.IntText(value=50, description="Rows:")
    audit = w.Checkbox(value=False, description="Audit columns")
    refresh = w.Button(description="Refresh Bloomberg", icon="refresh")
    status, output = w.HTML(), w.Output()
    state = {"table": None, "errors": pd.DataFrame(), "loading": False}

    def redraw(*_):
        if state["loading"] or state["table"] is None:
            return
        data = rank_table(state["table"], rank.value)
        if side.value != "ALL":
            data = data.loc[data["SIDE"].eq(side.value)]
        if search.value.strip():
            data = data.loc[data["TICKER"].str.contains(search.value.strip(), case=False, regex=False)]
        with output:
            output.clear_output(wait=True)
            display(HTML(f"<p>{data[rank.value].notna().sum()}/{len(data)} rows have the selected ranking metric. "
                         "Ranking is descending by that percentage, not by dollar premium.</p>"))
            display_table(data.head(max(1, limit.value)), audit.value)
            if not state["errors"].empty:
                display(HTML("<h4>Bloomberg query issues</h4>" + state["errors"].to_html(index=False, escape=True)))

    def load(*_):
        state["loading"] = True
        refresh.disabled = True
        output.clear_output()
        state["table"] = None
        try:
            table, errors = refresh_percentage_table(
                bq, selected, iv_units=iv_units,
                progress=lambda text: setattr(status, "value", escape(text)))
            state.update(table=table, errors=errors)
            status.value = f"Loaded {len(table)} selections; {len(errors)} query issues. IV input unit: {escape(iv_units)}."
        except Exception as exc:
            status.value = "Loading failed: " + escape(str(exc))
        finally:
            state["loading"] = False
            refresh.disabled = False
        redraw()

    for control in [rank, side, search, limit, audit]:
        control.observe(redraw, names="value")
    refresh.on_click(load)
    banner = w.HTML("<h3>Options | Implied volatility and premium (%)</h3>"
                    "<p>Existing contract selections are preserved. IV = Bloomberg IVOL(MID); "
                    "premium = (bid + ask) / 2. Premium percentages are not annualized. "
                    "IV is not independently recalibrated to this displayed midpoint. "
                    "Fetch timestamps are not quote timestamps. Standard USD equity-option units only. "
                    "This is a current-price patch, not a historical-chain or one-year-Z-score loader.</p>")
    ui = w.VBox([banner, w.HBox([rank, side, refresh]), w.HBox([search, limit, audit]), status, output])
    ui.data_state = state
    return ui


## Launch inside your existing BQNT notebook

Use the complete IDs held in your DataFrame, not a clipped ID copied from its display. Required columns: `TICKER`, `SIDE`, `EXPIRY`, `STRIKE`, `ID`. Column names are case-insensitive; underscores are normalized to spaces.

Run the cell below in the notebook where `result` already holds your existing selected-contract table, then click **Refresh Bloomberg**. Refresh updates market inputs for the same option IDs; it does not reselect expired contracts.

In [ ]:
import bql
from IPython.display import display

# result is the existing DataFrame that produced your option table.
option_rows = result
bq = bql.Service()

percent_dashboard = launch_dashboard(
    bq,
    option_rows,
    iv_units="percent",  # Raw 35 -> 35%; verify this unit against Bloomberg.
)
display(percent_dashboard)

## Non-widget alternative

This performs the same current-data requests and displays a table directly. The numeric result remains available for your own dashboard logic.

In [ ]:
# Uncomment to run without widgets.
# table, errors = refresh_percentage_table(bq, option_rows, iv_units="percent")
# table = rank_table(table, metric="IMPLIED VOL PCT")
# display_table(table)
# display(errors)

# Other rank metrics:
# "PREMIUM PCT SPOT" and "PREMIUM PCT STRIKE"

# After the widget has loaded, its unfiltered numeric table is available at:
# percent_dashboard.data_state["table"]

## Calculation reference

For premium P, spot S and strike K:

- Premium % spot = 100 P / S.
- Premium % strike = 100 P / K.
- Strike % spot = 100 K / S.
- Spread % = 100 (ask − bid) / P, when P > 0.

Using only the values supplied in the conversation: P = 131.4, S = 1712.89, K = 1540, bid = 128.1 and ask = 134.7. These imply 7.671% of spot, 8.532% of strike, 89.907% strike/spot and a 5.023% bid–ask spread relative to mid. No IV was supplied, so no IV value is fabricated.

Store percentages as numbers (7.671 means 7.671%) and append a literal percent sign for display. Formatting that same number with `:.3%` would multiply it by 100 again.

### Sources and API verification

The original IQMO BQuant OMON example uses the BQL `ivol(mid)` expression:
https://blog.iqmo.com/blog/bqnt/bquant_e_7_omonapp/

OCC/OIC explains annualized volatility and model-implied volatility:
https://www.optionseducation.org/referencelibrary/faq/technical-information

Pandas documents display formatting separately from stored numeric data:
https://pandas.pydata.org/docs/reference/api/pandas.io.formats.style.Styler.format.html

Public examples do not verify your Bloomberg entitlements, field units or current BQNT runtime. An IV query failure is displayed; valid premiums remain available without a model-based replacement IV.